In [1]:
import torch
import time

def get_tensor_size_mb(tensor):
    return tensor.element_size() * tensor.nelement() / (1024 * 1024)

def benchmark_transfer(tensor, n_iter=100, label="Tensor"):
    device = torch.device("cuda")
    
    # 데이터 크기 계산
    size_mb = get_tensor_size_mb(tensor)
    print(f"[{label}]")
    print(f" - Shape: {tensor.shape}")
    print(f" - Dtype: {tensor.dtype}")
    print(f" - Size : {size_mb:.2f} MB")

    # Warmup (초기화 오버헤드 제거)
    _ = tensor.to(device, non_blocking=True)
    torch.cuda.synchronize()

    # 시간 측정 설정
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    # 측정 시작
    start_event.record()
    for _ in range(n_iter):
        # CPU -> GPU 복사
        _ = tensor.to(device, non_blocking=True)
    end_event.record()
    
    # GPU 작업 완료 대기
    torch.cuda.synchronize()

    # 결과 계산
    total_time_ms = start_event.elapsed_time(end_event)
    avg_time_ms = total_time_ms / n_iter
    
    print(f" -> 평균 전송 시간: {avg_time_ms:.4f} ms")
    print(f" -> 대역폭 (Bandwidth): {(size_mb / avg_time_ms * 1000 / 1024):.2f} GB/s")
    print("-" * 40)

def main():
    if not torch.cuda.is_available():
        print("CUDA를 사용할 수 없습니다.")
        return

    # 1. 4096 * 4096 float16 Tensor 생성
    # pin_memory()를 사용하여 전송 속도 최적화 (실제 고성능 환경 시뮬레이션)
    tensor_fp16 = torch.randn(4096, 4096, dtype=torch.float16).pin_memory()

    # 2. 4096 * 2048 uint8 Tensor 생성ß
    tensor_uint8 = torch.randint(0, 255, (4096, 2048), dtype=torch.uint8).pin_memory()

    print("=== RAM to GPU Transfer Benchmark ===\n")
    
    benchmark_transfer(tensor_fp16, label="Case 1: Float16")
    benchmark_transfer(tensor_uint8, label="Case 2: Uint8")

if __name__ == "__main__":
    main()

=== RAM to GPU Transfer Benchmark ===

[Case 1: Float16]
 - Shape: torch.Size([4096, 4096])
 - Dtype: torch.float16
 - Size : 32.00 MB
 -> 평균 전송 시간: 1.5726 ms
 -> 대역폭 (Bandwidth): 19.87 GB/s
----------------------------------------
[Case 2: Uint8]
 - Shape: torch.Size([4096, 2048])
 - Dtype: torch.uint8
 - Size : 8.00 MB
 -> 평균 전송 시간: 0.3248 ms
 -> 대역폭 (Bandwidth): 24.06 GB/s
----------------------------------------
